# MIS Dashboard — Tienda de Moda
## ISID223 · Proyecto Primer Bimestre
**Empresa:** StyleTrend S.A.  
**Misión:** Transformar datos transaccionales (TPS) en inteligencia gerencial (MIS) para la toma de decisiones tácticas y estratégicas.

---


## Fase 1: Planificación y Análisis

### 1.1 Modelo de Negocio
**Empresa:** StyleTrend S.A. — Tienda de moda urbana  
**Misión:** Ofrecer ropa y accesorios de tendencia con excelente relación calidad-precio.  
**Visión:** Ser la tienda de moda urbana líder en ventas online y física en Ecuador al 2027.  
**Objetivos estratégicos:**
- Incrementar ingresos mensuales en 15% trimestral
- Aumentar el ticket promedio por cliente
- Fidelizar al Top 20 de clientes con mayor gasto

### 1.2 Stakeholders del MIS
| Stakeholder | Preguntas clave |
|---|---|
| Gerente General | ¿Cuál es el ingreso total? ¿Vamos bien vs. mes anterior? |
| Gerente de Ventas | ¿Cuáles son los productos más vendidos? ¿Quiénes son nuestros mejores clientes? |
| Gerente de Inventario | ¿Qué categorías rotan más? ¿Qué productos tienen bajo stock y alta demanda? |
| Gerente de Marketing | ¿Qué días venden más? ¿A qué clientes debemos fidelizar? |

### 1.3 Preguntas clave del negocio
1. ¿Cuáles son los 5 productos que más ingresos generan?
2. ¿Cuáles son los 5 clientes con mayor gasto total?
3. ¿Qué categoría de ropa domina las ventas?
4. ¿Cuál es el ticket promedio por transacción?
5. ¿Cómo evolucionan las ventas diariamente?
6. ¿Qué días de la semana se vende más?
7. ¿Qué productos están en bajo stock pero tienen alta demanda?
8. ¿Cuál es la distribución de ingresos por categoría?


## Fase 2: Definición de KPIs

| # | KPI | Fórmula | Stakeholder |
|---|-----|---------|-------------|
| 1 | Ingresos Totales | Σ ventas.total | Gerente General |
| 2 | Ticket Promedio | Ingresos Totales / N° Transacciones | Gerente de Ventas |
| 3 | Unidades Vendidas | Σ ventas.cantidad | Gerente de Inventario |
| 4 | Top 5 Productos (ingresos) | GROUP BY producto → SUM(total) | Gerente de Ventas |
| 5 | Top 5 Clientes (ingresos) | GROUP BY cliente → SUM(total) | Gerente de Marketing |
| 6 | Ingresos por Categoría | GROUP BY categoria → SUM(total) | Gerente General |
| 7 | Ventas Diarias | GROUP BY fecha → SUM(total) | Gerente General |
| 8 | Ventas por Día de Semana | GROUP BY dayofweek → SUM(total) | Gerente de Marketing |


## Fase 3: ETL e Implementación del Dashboard

### 3.1 Importar librerías

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo visual profesional
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#F8F9FA'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4
plt.rcParams['font.family'] = 'DejaVu Sans'

COLORES = ['#2D6A9F','#E07B39','#3BAB6F','#C0392B','#8E44AD',
           '#16A085','#D4AC0D','#1ABC9C','#E74C3C','#2980B9']
COLOR_PRIMARIO = '#2D6A9F'
COLOR_ACENTO   = '#E07B39'

print('✅ Librerías cargadas correctamente')

### 3.2 Extracción y limpieza (ETL)

In [ ]:
# ── EXTRACCIÓN ──────────────────────────────────────────
productos = pd.read_csv('productos.csv')
usuarios  = pd.read_csv('usuarios_100.csv')

# ventas: la fila 1 quedó pegada en el header → corrección
ventas_raw = pd.read_csv('ventas.csv', usecols=range(8))
ventas_raw.columns = ['idVenta','idUsuario','nombreUsuario',
                       'correoUsuario','idProducto','cantidad','total','fecha']
fila1 = pd.DataFrame(
    [[1, 33, 'Emily Ruiz', 'emily.ruiz87@gmail.com', 172, 1, 70.64, '2026-05-03']],
    columns=ventas_raw.columns)
ventas = pd.concat([fila1, ventas_raw], ignore_index=True)

# ── TRANSFORMACIÓN ──────────────────────────────────────
ventas['fecha']      = pd.to_datetime(ventas['fecha'])
ventas['total']      = pd.to_numeric(ventas['total'], errors='coerce')
ventas['dia_semana'] = ventas['fecha'].dt.day_name()
ventas['semana']     = ventas['fecha'].dt.isocalendar().week

# Join con productos para obtener categoría y nombre
ventas_full = ventas.merge(
    productos[['id','nombre','categoria','precio']],
    left_on='idProducto', right_on='id', how='left'
)

# ── VALIDACIÓN ──────────────────────────────────────────
print('── REPORTE DE CALIDAD ──────────────────────────────')
print(f'  Ventas totales   : {len(ventas):,}')
print(f'  Valores nulos    : {ventas.isnull().sum().sum()}')
print(f'  Rango fechas     : {ventas.fecha.min().date()} → {ventas.fecha.max().date()}')
print(f'  Productos únicos : {ventas.idProducto.nunique()}')
print(f'  Clientes únicos  : {ventas.idUsuario.nunique()}')
print(f'  Ingresos totales : $ {ventas.total.sum():,.2f}')
print('────────────────────────────────────────────────────')
print('✅ ETL completado')

### 3.3 Cálculo de KPIs

In [ ]:
# KPI 1: Ingresos Totales
ingresos_totales = ventas['total'].sum()

# KPI 2: Ticket Promedio
ticket_promedio = ingresos_totales / ventas['idVenta'].nunique()

# KPI 3: Unidades vendidas
unidades_vendidas = ventas['cantidad'].sum()

# KPI 4: Top 5 Productos por ingresos
top_productos = (ventas_full.groupby('nombre')['total']
                 .sum().sort_values(ascending=False).head(5))

# KPI 5: Top 5 Clientes
top_clientes = (ventas.groupby('nombreUsuario')['total']
                .sum().sort_values(ascending=False).head(5))

# KPI 6: Ingresos por Categoría
ing_categoria = (ventas_full.groupby('categoria')['total']
                 .sum().sort_values(ascending=False))

# KPI 7: Ventas Diarias
ventas_diarias = ventas.groupby('fecha')['total'].sum()

# KPI 8: Ventas por Día de Semana
orden_dias = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
nombres_dias = {'Monday':'Lunes','Tuesday':'Martes','Wednesday':'Miércoles',
                'Thursday':'Jueves','Friday':'Viernes','Saturday':'Sábado','Sunday':'Domingo'}
ventas_semana = (ventas.groupby('dia_semana')['total']
                 .sum().reindex(orden_dias).fillna(0))
ventas_semana.index = [nombres_dias[d] for d in ventas_semana.index]

print(f'KPI 1 — Ingresos Totales  : $ {ingresos_totales:,.2f}')
print(f'KPI 2 — Ticket Promedio   : $ {ticket_promedio:,.2f}')
print(f'KPI 3 — Unidades Vendidas : {unidades_vendidas:,}')
print()
print('KPI 4 — Top 5 Productos:')
print(top_productos.to_string())
print()
print('KPI 5 — Top 5 Clientes:')
print(top_clientes.to_string())

### 3.4 Dashboard 1 — Reporte Ejecutivo (KPIs + Tendencia)

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.suptitle('StyleTrend S.A. — Dashboard Ejecutivo | Mayo 2026',
             fontsize=18, fontweight='bold', color='#1A1A2E', y=0.98)

gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.4)

# ── Tarjetas KPI ──────────────────────────────────────
def tarjeta_kpi(ax, titulo, valor, subtitulo='', color='#2D6A9F'):
    ax.set_facecolor(color)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.text(0.5, 0.70, titulo, transform=ax.transAxes,
            ha='center', va='center', fontsize=11, color='white', alpha=0.85)
    ax.text(0.5, 0.38, valor, transform=ax.transAxes,
            ha='center', va='center', fontsize=20, fontweight='bold', color='white')
    ax.text(0.5, 0.10, subtitulo, transform=ax.transAxes,
            ha='center', va='center', fontsize=9, color='white', alpha=0.7)

ax_k1 = fig.add_subplot(gs[0, 0])
ax_k2 = fig.add_subplot(gs[0, 1])
ax_k3 = fig.add_subplot(gs[0, 2])
ax_k4 = fig.add_subplot(gs[0, 3])

tarjeta_kpi(ax_k1, 'Ingresos Totales', f'$ {ingresos_totales:,.0f}',
            'Mayo 2026', '#2D6A9F')
tarjeta_kpi(ax_k2, 'Ticket Promedio', f'$ {ticket_promedio:.2f}',
            'Por transacción', '#E07B39')
tarjeta_kpi(ax_k3, 'Transacciones', f'{len(ventas):,}',
            'Total en el período', '#3BAB6F')
tarjeta_kpi(ax_k4, 'Unidades Vendidas', f'{unidades_vendidas:,}',
            'Artículos despachados', '#8E44AD')

# ── Ventas Diarias ────────────────────────────────────
ax_line = fig.add_subplot(gs[1, :])
ax_line.fill_between(ventas_diarias.index, ventas_diarias.values,
                     alpha=0.15, color=COLOR_PRIMARIO)
ax_line.plot(ventas_diarias.index, ventas_diarias.values,
             color=COLOR_PRIMARIO, linewidth=2.2, marker='o', markersize=4)
ax_line.set_title('Evolución de Ventas Diarias', fontweight='bold', fontsize=12)
ax_line.set_ylabel('Ingresos ($)', fontsize=10)
ax_line.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%d %b'))
plt.setp(ax_line.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)
ax_line.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(
    lambda x, _: f'${x:,.0f}'))

# ── Ventas por Día de Semana ──────────────────────────
ax_sem = fig.add_subplot(gs[2, :2])
bars = ax_sem.bar(ventas_semana.index, ventas_semana.values,
                  color=COLORES[:7], edgecolor='white', linewidth=0.5)
ax_sem.set_title('Ingresos por Día de Semana', fontweight='bold', fontsize=12)
ax_sem.set_ylabel('Ingresos ($)', fontsize=10)
ax_sem.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(
    lambda x, _: f'${x:,.0f}'))
for bar in bars:
    ax_sem.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 100,
                f'${bar.get_height():,.0f}',
                ha='center', va='bottom', fontsize=7.5, fontweight='bold')

# ── Ingresos por Categoría (pie) ─────────────────────
ax_pie = fig.add_subplot(gs[2, 2:])
wedges, texts, autotexts = ax_pie.pie(
    ing_categoria.values,
    labels=None,
    autopct='%1.1f%%',
    colors=COLORES[:len(ing_categoria)],
    startangle=90, pctdistance=0.82,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.2})
for at in autotexts: at.set_fontsize(8)
ax_pie.set_title('Ingresos por Categoría', fontweight='bold', fontsize=12)
ax_pie.legend(ing_categoria.index, loc='center left',
              bbox_to_anchor=(1, 0.5), fontsize=8)

plt.savefig('dashboard_ejecutivo.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard Ejecutivo generado')

### 3.5 Dashboard 2 — Análisis de Ventas y Productos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('StyleTrend S.A. — Análisis de Ventas | Mayo 2026',
             fontsize=15, fontweight='bold', color='#1A1A2E')

# ── Top 5 Productos (barras horizontales) ─────────────
ax1 = axes[0]
prod_sorted = top_productos.sort_values()
bars = ax1.barh(range(len(prod_sorted)), prod_sorted.values,
                color=COLORES[:5], edgecolor='white', height=0.6)
ax1.set_yticks(range(len(prod_sorted)))
ax1.set_yticklabels([t[:22] for t in prod_sorted.index], fontsize=10)
ax1.set_title('Top 5 Productos por Ingresos', fontweight='bold', fontsize=12)
ax1.set_xlabel('Ingresos ($)')
ax1.xaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(
    lambda x, _: f'${x:,.0f}'))
for i, bar in enumerate(bars):
    ax1.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
             f'${bar.get_width():,.0f}', va='center', fontsize=9, fontweight='bold')

# ── Top 5 Clientes ─────────────────────────────────────
ax2 = axes[1]
cli_sorted = top_clientes.sort_values()
bars2 = ax2.barh(range(len(cli_sorted)), cli_sorted.values,
                 color=COLORES[5:10], edgecolor='white', height=0.6)
ax2.set_yticks(range(len(cli_sorted)))
ax2.set_yticklabels(cli_sorted.index, fontsize=10)
ax2.set_title('Top 5 Clientes por Ingresos', fontweight='bold', fontsize=12)
ax2.set_xlabel('Ingresos ($)')
ax2.xaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(
    lambda x, _: f'${x:,.0f}'))
for i, bar in enumerate(bars2):
    ax2.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
             f'${bar.get_width():,.0f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('dashboard_ventas.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard de Ventas generado')

### 3.6 Dashboard 3 — Inventario y Rotación

In [ ]:
# Calcular demanda por producto y cruzar con stock
demanda_prod = (ventas_full.groupby(['idProducto','nombre','categoria'])
                .agg(unidades_vendidas=('cantidad','sum'),
                     ingresos=('total','sum')).reset_index())

inventario = demanda_prod.merge(
    productos[['id','stock']], left_on='idProducto', right_on='id', how='left')

# Riesgo: alto demanda + bajo stock
inventario['riesgo'] = (inventario['unidades_vendidas'] /
                        (inventario['stock'] + 1))  # +1 para evitar div/0

top_riesgo = inventario.nlargest(10, 'riesgo')
top_rotacion = inventario.nlargest(5, 'unidades_vendidas')
ing_cat = inventario.groupby('categoria')['ingresos'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('StyleTrend S.A. — Inventario y Rotación | Mayo 2026',
             fontsize=15, fontweight='bold', color='#1A1A2E')

# ── Productos en riesgo de quiebre de stock ───────────
ax1 = axes[0]
colores_riesgo = ['#C0392B' if r > 5 else '#E07B39' if r > 2 else '#3BAB6F'
                  for r in top_riesgo['riesgo']]
bars = ax1.barh(range(len(top_riesgo)),
                top_riesgo['riesgo'].values,
                color=colores_riesgo, edgecolor='white', height=0.6)
ax1.set_yticks(range(len(top_riesgo)))
ax1.set_yticklabels([n[:20] for n in top_riesgo['nombre']], fontsize=8)
ax1.set_title('Riesgo Quiebre de Stock\n(Demanda / Stock disponible)',
              fontweight='bold', fontsize=11)
ax1.set_xlabel('Índice de Riesgo')
parches = [
    mpatches.Patch(color='#C0392B', label='Alto riesgo (>5)'),
    mpatches.Patch(color='#E07B39', label='Riesgo medio (2-5)'),
    mpatches.Patch(color='#3BAB6F', label='Bajo riesgo (<2)'),
]
ax1.legend(handles=parches, fontsize=7, loc='lower right')

# ── Ingresos por categoría (barras) ──────────────────
ax2 = axes[1]
bars2 = ax2.bar(ing_cat.index, ing_cat.values,
                color=COLORES[:len(ing_cat)], edgecolor='white')
ax2.set_title('Ingresos por Categoría', fontweight='bold', fontsize=11)
ax2.set_ylabel('Ingresos ($)')
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=40, ha='right', fontsize=8)
ax2.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(
    lambda x, _: f'${x:,.0f}'))

# ── Top 5 por unidades vendidas ──────────────────────
ax3 = axes[2]
rot_sorted = top_rotacion.sort_values('unidades_vendidas')
bars3 = ax3.barh(range(len(rot_sorted)),
                 rot_sorted['unidades_vendidas'].values,
                 color=COLORES[:5], edgecolor='white', height=0.6)
ax3.set_yticks(range(len(rot_sorted)))
ax3.set_yticklabels([n[:22] for n in rot_sorted['nombre']], fontsize=9)
ax3.set_title('Top 5 Productos por Unidades\nVendidas',
              fontweight='bold', fontsize=11)
ax3.set_xlabel('Unidades vendidas')
for bar in bars3:
    ax3.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             str(int(bar.get_width())), va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('dashboard_inventario.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard de Inventario generado')

---
## Informe Escrito — Propuesta de Valor del MIS

### Resumen Ejecutivo
StyleTrend S.A. opera con datos transaccionales abundantes pero hasta ahora sin visibilidad gerencial. Este MIS transforma esos datos en inteligencia accionable.

### Hallazgos Clave del Análisis
| Métrica | Valor |
|---------|-------|
| Ingresos totales (Mayo 2026) | $171,861.50 |
| Ticket promedio por transacción | $55.44 |
| Categoría líder | Jeans (15.1% de ingresos) |
| Producto estrella | Pantalón Cargo Verde ($2,330) |
| Cliente top | Maria Gomez ($2,536) |

### Ventaja Competitiva del MIS
1. **Visibilidad en tiempo real:** La gerencia pasa de "operar a ciegas" a tomar decisiones basadas en datos.
2. **Identificación de clientes VIP:** El Top 5 de clientes concentra el 7.2% de los ingresos totales — son candidatos prioritarios para programas de fidelización.
3. **Prevención de quiebre de stock:** El índice de riesgo detecta automáticamente productos con alta demanda y bajo inventario antes de que se agoten.
4. **Optimización de marketing:** Conocer los días con mayor y menor venta permite programar campañas en momentos estratégicos.

### Recomendaciones Gerenciales
- **Campaña de fidelización** al Top 20 de clientes por ingresos.
- **Reposición urgente** de productos con índice de riesgo > 5.
- **Enfoque de marketing** en Jeans y Chaquetas, las categorías de mayor ingreso.
- **Promociones en días bajos** para nivelar la demanda semanal.
